# 03. MRI Brain Tumor Classification


## What This Notebook Does

This notebook shows a very basic image-classification pipeline:

1. read MRI image paths
2. turn each image into numbers
3. scale the numbers
4. reduce the feature size with PCA
5. train one SVM model
6. save the trained files


## Step 1: Setup


In [ ]:
from pathlib import Path
import sys

current_dir = Path.cwd().resolve()
possible_dirs = [current_dir, current_dir / "NeuroSense" / "notebooks"]
notebooks_dir = next((path for path in possible_dirs if path.exists() and path.name == "notebooks"), None)
if notebooks_dir is None:
    raise FileNotFoundError("Start Jupyter from the project root or from NeuroSense/notebooks.")

if str(notebooks_dir) not in sys.path:
    sys.path.insert(0, str(notebooks_dir))

from notebook_support import bootstrap_notebook

ctx = bootstrap_notebook()
DATASETS_DIR = ctx["datasets_dir"]
ARTIFACTS_DIR = ctx["artifacts_dir"]
CACHE_DIR = ctx["cache_dir"]
RANDOM_STATE = ctx["random_state"]

print("Datasets directory:", DATASETS_DIR)
print("Artifacts directory:", ARTIFACTS_DIR)


## Step 2: Import The Libraries


In [ ]:
import json
import joblib
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC

from notebook_support import collect_labeled_image_paths, extract_feature_dataset, label_counts, resolve_mri_directories
from utils.preprocessors import preprocess_mri_image


## Step 3: Collect Image Paths

The dataset already has separate Training and Testing folders, so we use them directly.


In [ ]:
def normalize_mri_label(folder_name):
    folder_name = folder_name.lower()
    if folder_name == "notumor":
        return "no_tumor"
    return folder_name


train_dir, test_dir = resolve_mri_directories(DATASETS_DIR)
train_records = collect_labeled_image_paths(train_dir, normalize_mri_label)
test_records = collect_labeled_image_paths(test_dir, normalize_mri_label)

print("Training images:", len(train_records))
print("Testing images:", len(test_records))
print("Train labels:", label_counts(label for _, label in train_records))
print("Test labels:", label_counts(label for _, label in test_records))


## Step 4: Turn Images Into Feature Vectors

SVM needs numeric vectors, so each MRI image is converted into a flat grayscale vector. Then we scale the values and apply PCA.


In [ ]:
X_train_raw, y_train_raw = extract_feature_dataset(
    train_records,
    preprocess_mri_image,
    cache_path=CACHE_DIR / "mri_train_features.npz",
    progress_interval=250,
)
X_test_raw, y_test_raw = extract_feature_dataset(
    test_records,
    preprocess_mri_image,
    cache_path=CACHE_DIR / "mri_test_features.npz",
    progress_interval=250,
)

encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train_raw)
y_test = encoder.transform(y_test_raw)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled = scaler.transform(X_test_raw)

pca = PCA(n_components=100)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print("Train feature shape:", X_train_raw.shape)
print("Test feature shape:", X_test_raw.shape)
print("PCA output shape:", X_train_pca.shape)
print("Explained variance:", round(float(pca.explained_variance_ratio_.sum()), 4))


## Step 5: Train The Model


In [ ]:
model = SVC(kernel="rbf", C=5.0, probability=True, random_state=RANDOM_STATE)
model.fit(X_train_pca, y_train)
y_pred = model.predict(X_test_pca)


## Step 6: Check The Result


In [ ]:
test_accuracy = accuracy_score(y_test, y_pred)
print("MRI test accuracy:", round(test_accuracy, 4))
print(classification_report(y_test, y_pred, target_names=encoder.classes_))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    display_labels=encoder.classes_,
    cmap="Greens",
    xticks_rotation=20,
)
plt.title("MRI confusion matrix")
plt.tight_layout()
plt.show()

figure, axes = plt.subplots(1, 3, figsize=(9, 3))
for axis, sample, label in zip(axes, X_test_raw[:3], y_test_raw[:3]):
    axis.imshow(sample.reshape(64, 64), cmap="gray")
    axis.set_title(label)
    axis.axis("off")
plt.tight_layout()
plt.show()


## Step 7: Save The Trained Files


In [ ]:
artifact_dir = ARTIFACTS_DIR / "mri"
artifact_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(model, artifact_dir / "mri_model.pkl")
joblib.dump(scaler, artifact_dir / "mri_scaler.pkl")
joblib.dump(pca, artifact_dir / "mri_pca.pkl")
joblib.dump(encoder, artifact_dir / "mri_label_encoder.pkl")

metadata = {
    "data_source": "Public Dataset (Kaggle/Brain-Tumor)",
    "data_source_note": "MRI images are read from the dataset's Training and Testing folders.",
    "evaluation_method": "Fixed Train/Test Split (Folders)",
    "test_accuracy": round(float(test_accuracy), 4),
    "pca_components": 100,
}
with open(artifact_dir / "mri_metadata.json", "w") as file:
    json.dump(metadata, file, indent=2)

print("Saved MRI files to:", artifact_dir)
